# Monitoring and Logging for ProductionStructured logging, metrics, and alerts> Converted from `01_monitoring.py` - part of **05 Production and Operations**.

## Setup

In [ ]:
# ============ IMPORTS AND SETUP ===========================================import loggingimport jsonimport timefrom datetime import datetime, timezonefrom functools import wrapsfrom typing import Any, Callablefrom langchain_openai import ChatOpenAIfrom langchain_core.callbacks import BaseCallbackHandlerfrom langchain_core.messages import HumanMessagefrom langsmith import traceablefrom dotenv import load_dotenvload_dotenv()# === Structured Logging ===

### `JSONFormatter`Format logs as JSON for log aggregation.

In [ ]:
# ============ JSONFORMATTER ===============================================class JSONFormatter(logging.Formatter):    """Format logs as JSON for log aggregation."""    def format(self, record):        log_obj = {            "timestamp": datetime.now(timezone.utc).isoformat(),            "level": record.levelname,            "message": record.getMessage(),            "module": record.module,            "function": record.funcName,        }        if hasattr(record, "extra_data"):            log_obj.update(record.extra_data)        return json.dumps(log_obj)

### `setup_logging`Setup structured JSON logging.

In [ ]:
# ============ SETUP_LOGGING ===============================================def setup_logging():    """Setup structured JSON logging."""    logger = logging.getLogger("langgraph_app")    logger.setLevel(logging.INFO)    handler = logging.StreamHandler()    handler.setFormatter(JSONFormatter())    logger.addHandler(handler)    return logger

### `MetricsCollector`Collect and aggregate metrics.

In [ ]:
# ============ METRICSCOLLECTOR ============================================class MetricsCollector:    """Collect and aggregate metrics."""    def __init__(self):        self.metrics = {            "requests_total": 0,            "errors_total": 0,            "latency_sum": 0,            "latency_count": 0,            "tokens_input": 0,            "tokens_output": 0,            "cache_hits": 0,            "cache_misses": 0,        }    def record_request(        self,        latency_ms: float,        input_tokens: int,        output_tokens: int,        error: bool = False,        cache_hit: bool = False,    ):        self.metrics["requests_total"] += 1        self.metrics["latency_sum"] += latency_ms        self.metrics["latency_count"] += 1        self.metrics["tokens_input"] += input_tokens        self.metrics["tokens_output"] += output_tokens        if error:            self.metrics["errors_total"] += 1        if cache_hit:            self.metrics["cache_hits"] += 1        else:            self.metrics["cache_misses"] += 1    def get_summary(self) -> dict:        avg_latency = (            self.metrics["latency_sum"] / self.metrics["latency_count"]            if self.metrics["latency_count"] > 0            else 0        )        error_rate = (            self.metrics["errors_total"] / self.metrics["requests_total"]            if self.metrics["requests_total"] > 0            else 0        )        cache_hit_rate = (            self.metrics["cache_hits"]            / (self.metrics["cache_hits"] + self.metrics["cache_misses"])            if (self.metrics["cache_hits"] + self.metrics["cache_misses"]) > 0            else 0        )        return {            "total_requests": self.metrics["requests_total"],            "total_errors": self.metrics["errors_total"],            "error_rate": f"{error_rate:.2%}",            "avg_latency_ms": round(avg_latency, 2),            "total_input_tokens": self.metrics["tokens_input"],            "total_output_tokens": self.metrics["tokens_output"],            "cache_hit_rate": f"{cache_hit_rate:.2%}",        }

### `InstrumentedLLM`LLM with full instrumentation.

In [ ]:
# ============ INSTRUMENTEDLLM =============================================class InstrumentedLLM:    """LLM with full instrumentation."""    def __init__(self):        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)        self.metrics = MetricsCollector()        self.logger = setup_logging()    @traceable(name="instrumented_invoke")    def invoke(self, query: str) -> str:        start_time = time.time()        error = False        try:            response = self.llm.invoke(query)            result = response.content            # Estimate tokens            input_tokens = len(query.split()) * 4 // 3            output_tokens = len(result.split()) * 4 // 3            self.metrics.record_request(                latency_ms=(time.time() - start_time) * 1000,                input_tokens=input_tokens,                output_tokens=output_tokens,                error=False,                cache_hit=False,            )            self.logger.info(                "LLM request completed",                extra={                    "extra_data": {                        "latency_ms": (time.time() - start_time) * 1000,                        "input_tokens": input_tokens,                        "output_tokens": output_tokens,                    }                },            )            return result        except Exception as e:            error = True            self.metrics.record_request(                latency_ms=(time.time() - start_time) * 1000,                input_tokens=0,                output_tokens=0,                error=True,                cache_hit=False,            )            self.logger.error(                f"LLM request failed: {e}", extra={"extra_data": {"error": str(e)}}            )            raise

### `demo_monitoring`Demonstrate monitoring.

In [ ]:
# ============ DEMO_MONITORING =============================================def demo_monitoring():    """Demonstrate monitoring."""    llm = InstrumentedLLM()    print("Monitoring Demo:\n")    queries = [        "What is Python?",        "Explain machine learning.",        "What is 2 + 2?",    ]    for query in queries:        result = llm.invoke(query)        print(f"Query: {query[:30]}... -> {result[:30]}...")    print("\nMetrics Summary:")    summary = llm.metrics.get_summary()    for key, value in summary.items():        print(f"  {key}: {value}")

## RunThe original `__main__` guard, kept verbatim. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is. Uncomment a line to run that demo.

In [ ]:
# ============ RUN =========================================================if __name__ == "__main__":    # logger = setup_logging()    # logger.info("Logging setup complete", extra={"extra_data": {"app": "langgraph"}})    demo_monitoring()

## SummaryDefined in this notebook:- `JSONFormatter()`- `setup_logging()`- `MetricsCollector()`- `InstrumentedLLM()`- `demo_monitoring()`